# MTCG: Multi-Template Computational Graph for Traffic Demand Flow Estimation

**Sioux Falls Network Case Study**

**Authors:**
- Xin (Bruce) Wu" Department of Civil and Environmental Engineering, Villanova University, USA
- Feng Shao" School of Mathematics, China University of Mining and Technology, China

**Contact:** xwu03@villanova.edu

---

MIT License

Copyright (c) 2026 Xin (Bruce) Wu, Feng Shao

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

---

## 1. Import Libraries

This notebook implements the **Multi-Template Computational Graph (MTCG)** for the Sioux Falls network case study.

The MTCG is a computational-graph-based deep learning framework for Dynamic Traffic Demand Flow Estimation (D-TDFE). It reformulates the classical OD matrix estimation and stochastic traffic assignment as a differentiable, layered architecture trainable via backpropagation.

**Dependencies:** PyTorch (deep learning), NetworkX (graph/path enumeration), NumPy/Pandas (data), Matplotlib (plotting).

In [345]:
import networkx as nx
import copy as cp
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from pandas import DataFrame
import scipy.sparse as sp
import math
import time

plt.rc('font',family='Times New Roman')

import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module
import torch.nn.functional as F
import torch.optim as optim

In [346]:
# --- Reproducibility ---
import torch
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Random seed set to {SEED}")

# ========== CONFIG ==========
S = 4    # Number of templates/channels
k = 5    # Number of shortest paths per OD pair
w1 = 1     # Link flow loss weight
w2 = 0.8   # Demand conservation loss weight
w3 = 0.01  # VDF regularization weight
w4 = 0     # Per-timestep OD demand loss weight
# ============================
print(f"S={S}, k={k}, w1={w1}, w2={w2}, w3={w3}, w4={w4}")


Random seed set to 42
S=4, k=5, w1=1, w2=0.8, w3=0.01, w4=0


## 2. Load Data

Load the Sioux Falls network data:
- **OD demand** (`demand.csv`): $\mathbf{q} \in \mathbb{R}^{1 \times |\mathcal{W}|}$, where $|\mathcal{W}| = 96$ OD pairs
- **OD pair indices** (`od_pair.csv`): origin and destination node IDs
- **Link flow** (`link_flow.csv`): observed link flows $\bar{\mathbf{v}}^m \in \mathbb{R}^{1 \times |\mathcal{A}|}$, where $|\mathcal{A}| = 76$ links
- **Link attributes** (`link_attributes.csv`): free-flow travel time $t_a^0$ and capacity $c_a$

The dataset contains 500 samples (days) with 12 timesteps each.

In [347]:
od_demand = pd.read_csv("data/demand.csv").values
od_pair = pd.read_csv("data/od_pair.csv").values

origin = od_pair[:,0]
destination = od_pair[:,1]
num_od = len(origin) # Number of OD pairs

od_demand = np.reshape(od_demand, [12,500,num_od]) # 12 timesteps, 500 days, 96 OD pairs
od_demand.shape

(12, 500, 96)

## 2.1 Inspect Mean OD Demand

Quick inspection of mean OD demand across samples and timesteps.

In [348]:
od_demand.mean(axis=-1).mean(axis=-1)

array([ 831.4521875,  891.4521875,  951.4521875, 1011.4521875,
       1071.4521875, 1131.4521875, 1101.4521875, 1041.4521875,
        981.4521875,  921.4521875,  861.4521875,  801.4521875])

## 2.2 Set Timesteps and Aggregate Demand

Set the number of timesteps $T = 8$ (covering the study period 7:00â€“9:00 AM).

The total OD demand over all timesteps serves as the reference input to the model:

$$\bar{\mathbf{q}} = \sum_{t=1}^{T} \mathbf{q}_t$$

Note: only the aggregate demand $\bar{\mathbf{q}}$ is available to the model; timestep-level OD demands are not directly observed.

In [349]:
T = 8 # timestep

demand_total = od_demand[:T].sum(axis=0)
demand_total = torch.from_numpy(demand_total).to(torch.float32)

demand_reference = od_demand[:T]
demand_reference = torch.from_numpy(demand_reference).to(torch.float32)

print(demand_total.shape)

torch.Size([500, 96])


## 2.3 Load Link Flows

Load observed link flows obtained via Stochastic User Equilibrium (SUE) assignment. These serve as ground truth $\bar{\mathbf{v}}_t^m$ for training and evaluation.

Shape: `(timesteps, samples, links)` = $(12, 500, 76)$, truncated to $T = 8$ timesteps.

In [350]:
# link flow
flow = pd.read_csv("data/link_flow.csv").values

num_link = flow.shape[1] # Number of links
flow = np.reshape(flow, [12,-1,num_link]) # 15 timesteps, 500 days, 76 links

link_flow = flow[:T]
link_flow = torch.from_numpy(link_flow).to(torch.float32)

print(link_flow.shape)

torch.Size([8, 500, 76])


## 2.4 Load Link Attributes

Load link-level attributes:
- **Free-flow travel time** $t_a^0$: the travel time on link $a$ under zero-flow conditions
- **Link capacity** $c_a$: used in the BPR volume-delay function

These are used in the Bureau of Public Roads (BPR) function to compute flow-dependent travel times.

In [351]:
link_attributes = pd.read_csv("data/link_attributes.csv")

# free flow travel time
free_flow_tt = link_attributes['free flow travel time'].values
free_flow_tt = torch.from_numpy(free_flow_tt).to(torch.float32)

# link capacity
link_capacity = link_attributes['link capacity'].values
link_capacity = torch.from_numpy(link_capacity).to(torch.float32)

link_attributes

,link number,start,end,free flow travel time,link capacity
0,1,1,2,3.6,2000
1,2,1,3,2.4,2000
2,3,2,1,3.6,2000
3,4,2,6,3.0,2000
4,5,3,1,2.4,2000
...,...,...,...,...,...
71,72,23,22,2.4,2000
72,73,23,24,1.2,2000
73,74,24,13,2.4,2000
74,75,24,21,1.8,2000


## 3. K-Shortest Paths Algorithm

Enumerate candidate paths for each OD pair using **Yen's K-shortest paths algorithm**.

For each OD pair $w \in \mathcal{W}$, $k = 5$ shortest paths are generated based on free-flow travel times. These paths form the candidate route set $\mathcal{R}_w$, which defines the path-link incidence matrix $\boldsymbol{\delta}$.

The path set is fixed during training (not dynamically updated).

In [352]:
# k shortest path
# Source: https://github.com/Mokerpoker/k_shortest_paths
def k_shortest_paths(G, source, target, k, weight = 'weights'):
   
    A = [nx.dijkstra_path(G, source, target, weight = 'weights')]
    A_len = [sum([G[A[0][l]][A[0][l + 1]]['weights'] for l in range(len(A[0]) - 1)])]
    B = []

    for i in range(1, k):
        for j in range(0, len(A[-1])-1):
            Gcopy = cp.deepcopy(G)
            spurnode = A[-1][j]
            rootpath = A[-1][:j + 1]
            for path in A:
                if rootpath == path[0:j + 1]:
                    if Gcopy.has_edge(path[j], path[j + 1]):
                        Gcopy.remove_edge(path[j], path[j + 1])
                    if Gcopy.has_edge(path[j + 1], path[j]):
                        Gcopy.remove_edge(path[j + 1], path[j])
            for n in rootpath:
                if n != spurnode:
                    Gcopy.remove_node(n)
            try:
                spurpath = nx.dijkstra_path(Gcopy, spurnode, target, weight = 'weights')
                totalpath = rootpath + spurpath[1:]
                if totalpath not in B:
                    B += [totalpath]
            except nx.NetworkXNoPath:
                continue
        if len(B) == 0:
            break
        lenB = [sum([G[path[l]][path[l + 1]]['weights'] for l in range(len(path) - 1)]) for path in B]
        B = [p for _,p in sorted(zip(lenB, B))]
        A.append(B[0])
        A_len.append(sorted(lenB)[0])
        B.remove(B[0])
    A_link = []
    for path in A:
        path_link = []
        for j in range(len(path)-1):
            link = [path[j], path[j+1]]
            link_number = edge_list.index(link)+1
            path_link.append(link_number)
        A_link.append(path_link)

    return A_link, A_len

## 3.1 Construct Network Graph and Generate Paths

Build the Sioux Falls network as a directed graph using NetworkX, then run the K-shortest paths algorithm for all 96 OD pairs.

Each path is stored as a sequence of link indices, which will be used to construct the path-link incidence matrix.

In [353]:
# Construct the transportation network graph
edge_list = link_attributes[['start','end']].values
edge_list = edge_list.tolist()
edges = pd.DataFrame()
edges['sources'] = np.array(edge_list)[:,0]
edges['targets'] = np.array(edge_list)[:,1]

## 3.2 Build Path-Link Incidence Matrix

Construct the **path-link incidence matrix** $\boldsymbol{\delta} \in \{0,1\}^{|\mathcal{R}| \times |\mathcal{A}|}$, stored as the tensor `LP` with shape $(S, |\mathcal{A}|, |\mathcal{R}|)$.

- $S$: number of templates (channels). Each template may use a different path set.
- For template $s = 1$: paths are generated using free-flow travel times.
- For templates $s > 1$: paths are generated using perturbed travel times ($t_a^0 + |\mathcal{N}(1,1)|$), producing different candidate route sets to represent different contextual archetypes.

The path-link incidence matrix is the core structural element of the CG, enabling the forward pass:

$$\mathbf{v} = \mathbf{f} \, \boldsymbol{\delta}$$

In [354]:
# S and k are set in the config cell above

# link path incidence tensor
LP = np.zeros([S, num_link, num_od*k])

for s in range(S):
    
    # Record link path matrix
    LP_s = np.zeros([num_link,1])

    for i in range(num_od):
        
        ori = origin[i]
        des = destination[i]

        if s<1:
            edges["weights"] = free_flow_tt
        else:
            # Randomly alter the travel time on links
            edges["weights"] = free_flow_tt + np.abs(np.random.normal(1,1,num_link))
            
        G = nx.from_pandas_edgelist(edges, source="sources", target="targets", edge_attr="weights", create_using=nx.DiGraph())
        
        path, path_time = k_shortest_paths(G, ori, des, k, weight = "weights")

        # The path contains k routes
        for p in path:

            LP0 = np.zeros([num_link,1])
            LP0[np.array(p)-1] = 1

            # Add a new path
            LP_s = np.hstack((LP_s,LP0))

    LP_s = np.delete(LP_s, 0, axis=1)   # shape: num_link x num_path
    LP[s] = LP_s

LP = torch.from_numpy(LP).to(torch.float32)
print(LP.shape)


C:\Users\xwu03\AppData\Local\Temp\ipykernel_45908\622975461.py:20: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  edges["weights"] = free_flow_tt + np.abs(np.random.normal(1,1,num_link))


torch.Size([4, 76, 480])


## 4. Train/Test Split

Split the 500 samples into:
- **Training set**: first 400 samples
- **Test set**: remaining 100 samples

Inputs: aggregate OD demand $\bar{\mathbf{q}} \in \mathbb{R}^{\text{batch} \times |\mathcal{W}|}$

Outputs: link flows $\bar{\mathbf{v}}_t \in \mathbb{R}^{T \times \text{batch} \times |\mathcal{A}|}$

In [355]:
# train set: the first 400 sets of data
train_input = demand_total[:400,:].to(torch.float32)
train_output = link_flow[:,:400,:].to(torch.float32)

# test set: the remaining 100 sets of data
test_input = demand_total[400:,:].to(torch.float32)
test_output = link_flow[:,400:,:].to(torch.float32)

print(train_input.shape)
print(train_output.shape)
print(test_input.shape)
print(test_output.shape)

torch.Size([400, 96])
torch.Size([8, 400, 76])
torch.Size([100, 96])
torch.Size([8, 100, 76])


## 5. BPR Volume-Delay Function

Define the **Bureau of Public Roads (BPR) function**, which computes flow-dependent link travel times:

$$t_a(v_a) = t_a^0 \left(1 + 0.15 \left(\frac{v_a}{c_a}\right)^4\right)$$

where $t_a^0$ is the free-flow travel time, $v_a$ is the link flow, and $c_a$ is the link capacity. This function is used within the CG to update travel times after each assignment step.

In [356]:
# Define the BPR function
def BPR(link_flow, free_flow_tt, link_capacity):
    link_time = free_flow_tt * (1 + 0.15 * (link_flow/link_capacity)**4)
    return link_time

## 5.1 Logit-Based Stochastic User Equilibrium Assignment

Define the **assignment function** implementing the Logit-based SUE model.

Given OD demand $\mathbf{q}$, value-of-time parameter $\boldsymbol{\theta}$, and path travel times, the route choice probability for path $r$ of OD pair $w$ is:

$$\rho_{wr} = \frac{\exp\left(-\theta_w \sum_{a \in \mathcal{A}} \delta_{ra} \, t_a\right)}{\sum_{r' \in \mathcal{R}_w} \exp\left(-\theta_w \sum_{a \in \mathcal{A}} \delta_{r'a} \, t_a\right)}$$

The three-layer CG forward pass is then:

1. **OD layer â†’ Path layer:** $\mathbf{f} = \mathbf{q} \, \boldsymbol{\rho}$ (path flows = demand Ã— route choice probabilities)
2. **Path layer â†’ Link layer:** $\mathbf{v} = \mathbf{f} \, \boldsymbol{\delta}$ (link flows = path flows Ã— incidence matrix)

The function also computes updated link travel times via BPR for the recurrent feedback loop.

In [357]:
# Given theta, path travel times, and the link-path incidence matrix, 
# the OD demands are distributed to obtain link flows and link travel times.
def assignment(demand, theta, path_time, LP):
    
    batch_size = demand.shape[0]
    
    # The OD demand assignment for the first timestep is based on the free flow travel time, while for subsequent timesteps, 
    # it is allocated based on the travel time from the previous timestep.
    if path_time.dim()==1:
        path_time = path_time.repeat(batch_size,1)
    
    # Reshaping the demand to have the same dimension as the path choice proportion matrix so that they can be multiplied.
    demand_expand = demand.reshape(-1,1).repeat(1,k).reshape(batch_size,-1) # shape: batch_size Ã— num_path
    
     # Reshape the theta
    theta_expand = theta.repeat(k,1).transpose(0,1).reshape(1,-1)   # shape: 1 Ã— num_path
    
    # Calculate e^(-theta*path_time)
    tp0 = (-path_time * theta_expand).reshape(batch_size,-1,k)  #shape: batch_sizeÃ—num_ODÃ—k
    tp = math.e ** tp0  #shape: batch_sizeÃ—num_ODÃ—k
    
    # Set a lower bound for e^(-theta*path_time)
    P0 = torch.max(tp, (10**-10)*torch.ones(tp.shape)) #shape: batch_sizeÃ—num_ODÃ—k

    # Sum over the last dimension
    P0_sum = torch.sum(P0,dim=2).view(batch_size,-1,1)  #shape: batch_sizeÃ—num_ODÃ—1

    # Calculate the path choice proportion
    P0 = P0/P0_sum #shape: batch_sizeÃ—num_ODÃ—k

    P = P0.view(batch_size,-1)  # shape: batch_size Ã— num_path

    path_flow = demand_expand * P  # shape: batch_size Ã— num_path

    # Estimate link flow and link time
    link_flow = torch.mm(path_flow,LP.T)   # shape: batch_size Ã— num_link
    link_time = BPR(link_flow, free_flow_tt, link_capacity)
    
    return link_flow,link_time

## 5.2 Parameter Initialization

Initialize all learnable parameters using **Kaiming (He) normal initialization**, which is well-suited for layers with ReLU/LeakyReLU activations.

In [358]:
def Initialization(parameters):
    for para in parameters:
        nn.init.kaiming_normal_(para)

## 6. Cross-Attention Mechanism

Implement the **cross-attention fusion** mechanism that learns sample-specific template weights $\lambda_{s,t}^m$.

The attention mechanism computes:

$$\lambda_{s,t}^m = \frac{\exp(e_{s,t}^m)}{\sum_{s'=1}^{S} \exp(e_{s',t}^m)}, \qquad e_{s,t}^m = \frac{(\bar{\mathbf{q}}^m\,\mathbf{U}^Q)(\mathbf{v}_{s,t}\,\mathbf{U}^K)^\top}{\sqrt{d_k}}$$

where:
- **Query** $\bar{\mathbf{q}}^m$: the aggregate reference OD demand for sample $m$, projected via a learned matrix $\mathbf{U}^Q$ + LayerNorm
- **Keys** $\mathbf{v}_{s,t}$: template-specific estimated link flows (no $m$ superscript — these are model outputs), projected via learned per-timestep matrices $\mathbf{U}^K$ + LayerNorm
- **Values**: the template-specific estimated link flows $\mathbf{v}_{s,t}$ and OD demands $\mathbf{q}_{s,t}$

The attention weights $\lambda_{s,t}^m$ are sample-specific (they depend on the query $\bar{\mathbf{q}}^m$) and directly interpretable as the contribution of each contextual archetype to the current traffic state.

In [359]:
# Calculate the attention coefficients between estimated link flows from each channel and input OD demands, 
# to obtain weighted values for link flows.

# Hyperparameter: hidden_size, the dimensions of the embedding tensor
class Attention(nn.Module):
    
    def __init__(self, num_od, num_link, hidden_size):
        
        super(Attention, self).__init__()
    
        #self.linear_demand = nn.Linear(num_od, hidden_size,bias = False)
        self.weights_demand = Parameter(torch.FloatTensor(num_od, hidden_size))
        self.weights_linkflow = Parameter(torch.FloatTensor(T, num_link, hidden_size))
        self.hidden_size = hidden_size
        
        Initialization([self.weights_demand, self.weights_linkflow])
    
    def forward(self, demand, X):
        
        batch_size = len(demand) # demand shape: (batch_size, num_od)
        
        q = torch.nn.LayerNorm(self.hidden_size)(torch.mm(demand, self.weights_demand)) # (batch_size, hidden_size)
        q = q.reshape(1,batch_size,1,-1) # shape: (1, batch size, 1, hidden_size)
        q = q.repeat(T,1,1,1) # shape: (T, batch size, 1, hidden_size)
        
        w = self.weights_linkflow.reshape(T, 1, num_link, -1) # shape: (T, 1, num_link, hidden_size)
        w = w.repeat(1, batch_size, 1, 1) # shape: (T, batch size, num_link, hidden_size)
        
        # X.shape: (S, T, batch size, num_link)
        X = X.permute(1,2,0,3) # shape: (T, batch size, S, num_link)
        
        x =  torch.nn.LayerNorm(self.hidden_size)(torch.matmul(X, w))  # shape: (T, batch size, S, hidden_size)  
        
        scores = torch.matmul(q, x.transpose(2, 3))/math.sqrt(x.shape[-1]) # (T, batch size, 1, S)
        
        att = F.softmax(scores, dim=-1) # (T, batch size, 1, S)
        
        x_output = torch.matmul(att,X).reshape(T, batch_size,-1) # (T, batch size, 1, num_link)-->(T, batch size, num_link)
        
        return att, x_output

## 7. MTCG Model Architecture

Define the **Multi-Template Computational Graph (MTCG)** – the main model class.

The MTCG consists of the following components for each template $s$:

1. **Demand proportion DNN**: Maps aggregate demand $\bar{\mathbf{q}}^m$ to per-timestep proportions via softmax:
$$\mathbf{q}_{s,t} = \bar{\mathbf{q}}^m \cdot \text{softmax}(\text{DNN}(\bar{\mathbf{q}}^m))_t$$

2. **Three-layer CG assignment** at each timestep $t$:
$$\mathbf{v}_{s,t} = \mathbf{v}(\mathbf{f}(\mathbf{q}_{s,t}, \mathbf{P}(\boldsymbol{\theta}_{s,t})), \boldsymbol{\delta}_s)$$

3. **Recurrent update** connecting timestep $t$ to $t+1$:
   - Link flows $\mathbf{v}_{s,t}$ and OD travel times $\mathbf{t}_{s,t}$ are mapped through DNNs
   - Fused via concatenation and projection
   - Multiplicative gating: $\mathbf{q}_{s,t+1} = (1 + 0.5 \cdot \mathbf{g}_t) \cdot \mathbf{q}_{s,t} + \mathbf{h}_t$

4. **Cross-attention fusion** across $S$ templates to produce final estimates:
$$\mathbf{v}_t = \sum_{s=1}^{S} \lambda_{s,t}^m \, \mathbf{v}_{s,t}, \qquad \mathbf{q}_t = \sum_{s=1}^{S} \lambda_{s,t}^m \, \mathbf{q}_{s,t}$$

where the attention weights $\lambda_{s,t}^m$ are sample-specific (depending on $\bar{\mathbf{q}}^m$), while the template outputs $\mathbf{v}_{s,t}$ and $\mathbf{q}_{s,t}$ are estimated quantities without $m$ superscript.

In [360]:
class MRLN(Module):
    
    # hidden_layer: the number of neurons in the hidden layer of the DNN that 
    #               maps the link flows from the previous timestep to the OD demands in the next timestep.
    # att_hidden_size:the dimensions of the embedding tensor for attention calculation
    # hidden_layer2: the number of neurons in the hidden layer of the DNN that maps the initial OD demand 
    #                to the proportion of OD demands for the first timestep.

    def __init__(self, num_link, num_od, hidden_layer, att_hidden_size, hidden_layer2):
        
        super(MRLN, self).__init__()
        
        self.num_od = num_od
        
        # Same channel, same timestep, and same OD pair share a common theta
        self.theta = Parameter(torch.FloatTensor(S, T, num_od))
        nn.init.xavier_uniform_(self.theta)
        
        # The initial allocation of OD demands using a three-layer DNN
        self.weights_d1 = Parameter(torch.FloatTensor(S, num_od, hidden_layer2[0]))
        self.weights_d2 = Parameter(torch.FloatTensor(S, hidden_layer2[0], hidden_layer2[1]))
        self.weights_d3 = Parameter(torch.FloatTensor(S, hidden_layer2[1], T * num_od))
        
        self.bias_d1 = Parameter(torch.FloatTensor(S, hidden_layer2[0]))
        self.bias_d2 = Parameter(torch.FloatTensor(S, hidden_layer2[1]))
        self.bias_d3 = Parameter(torch.FloatTensor(S, T * num_od))
        
        Initialization([self.weights_d1, self.weights_d2, self.weights_d3, self.bias_d1, self.bias_d2, self.bias_d3])
        
        ######## ODME ########
        # For each channel, ODME is performed using DNNs
        # Different timesteps share weights and biases
        
        # flow network
        self.weights1_v = Parameter(torch.FloatTensor(S, num_link, hidden_layer[0]))
        self.weights2_v = Parameter(torch.FloatTensor(S, hidden_layer[0], hidden_layer[1]))
        self.weights3_v = Parameter(torch.FloatTensor(S, hidden_layer[1], num_od))
        
        self.bias1_v = Parameter(torch.FloatTensor(S, hidden_layer[0]))
        self.bias2_v = Parameter(torch.FloatTensor(S, hidden_layer[1]))
        self.bias3_v = Parameter(torch.FloatTensor(S, num_od))
        
        Initialization([self.weights1_v, self.weights2_v, self.weights3_v, self.bias1_v, self.bias2_v, self.bias3_v])
        
        # od travel time network
        self.weights1_t = Parameter(torch.FloatTensor(S, num_od, hidden_layer[0]))
        self.weights2_t = Parameter(torch.FloatTensor(S, hidden_layer[0], hidden_layer[1]))
        self.weights3_t = Parameter(torch.FloatTensor(S, hidden_layer[1], num_od))
        
        self.bias1_t = Parameter(torch.FloatTensor(S, hidden_layer[0]))
        self.bias2_t = Parameter(torch.FloatTensor(S, hidden_layer[1]))
        self.bias3_t = Parameter(torch.FloatTensor(S, num_od))
        
        Initialization([self.weights1_t, self.weights2_t, self.weights3_t, self.bias1_t, self.bias2_t, self.bias3_t])
              
        # fusion network
        self.weights_fusion = Parameter(torch.FloatTensor(S, 2 * num_od, num_od))
        self.bias_fusion = Parameter(torch.FloatTensor(S, num_od))
        
        Initialization([self.weights_fusion, self.bias_fusion])
        
        # flow network 2
        self.weights1_v2 = Parameter(torch.FloatTensor(S, num_od, num_od))
#         self.weights2_v2 = Parameter(torch.FloatTensor(S, hidden_layer[0], hidden_layer[1]))
#         self.weights3_v2 = Parameter(torch.FloatTensor(S, hidden_layer[1], num_od))
        
        self.bias1_v2 = Parameter(torch.FloatTensor(S, num_od))
#         self.bias2_v2 = Parameter(torch.FloatTensor(S, hidden_layer[1]))
#         self.bias3_v2 = Parameter(torch.FloatTensor(S, num_od))
        
        Initialization([self.weights1_v2, self.bias1_v2])
        
        ##################
        
        # Attention layer
        self.attention = Attention(num_od, num_link, att_hidden_size)

    def forward(self, demand):
        
        batch_size = demand.shape[0]
        
        # VOT
        theta = torch.abs(self.theta)
        
        # For storing link flows and OD demands
        link_flow = torch.zeros(S, T, batch_size, num_link)
        demand_pred = torch.zeros(S, T, batch_size, num_od) 
        demand_sum = torch.zeros(S, batch_size, num_od)
        
        for s in range(S):
            
            demand_proportion = torch.nn.LayerNorm(num_od)(demand)
            
            demand_proportion = F.leaky_relu(torch.mm(demand_proportion, self.weights_d1[s]) + self.bias_d1[s], negative_slope=0.2)
            demand_proportion = F.leaky_relu(torch.mm(demand_proportion, self.weights_d2[s]) + self.bias_d2[s], negative_slope=0.2)
            demand_proportion = F.leaky_relu(torch.mm(demand_proportion, self.weights_d3[s]) + self.bias_d3[s], negative_slope=0.2)

            demand_proportion = demand_proportion.reshape(batch_size, self.num_od, T)
#             demand_proportion = torch.nn.LayerNorm(T)(demand_proportion)
            demand_proportion = torch.softmax(demand_proportion, axis=2)  # (batch size, num_od, T)

            demand_initial = demand_proportion * demand.reshape(batch_size, self.num_od, 1)  # (batch size, num_od, T)
            demand_initial = demand_initial.permute(2,0,1)   # (T, batch size, num_od)
        
            # th 1th timestep
            demand_st = demand_initial[0]   # shape:(batch size, num_od)
            theta_st = theta[s, 0]
            LP_s = LP[s]
            
            link_time_st = free_flow_tt.repeat(batch_size,1)
            path_time_st = torch.mm(link_time_st,LP_s)
            
            link_flow_st,link_time_st = assignment(demand_st, theta_st, path_time_st, LP_s)
            
            # Calculate the od travel time
            path_time_st = torch.mm(link_time_st,LP_s) # shape:(batch size, num_path)
            od_travel_time = torch.mean(path_time_st.reshape(batch_size, num_od, k),dim=-1) # shape:(batch size, num_od)
            
            link_flow_s = torch.zeros(T, batch_size, num_link)
            link_flow_s[0] = link_flow_st
            
            demand_s = torch.zeros(T, batch_size, num_od)
            demand_s[0] = demand_st
            
            demand_sum_s = demand_st
            
            for t in range(T-1):
                                
                #########
                demand_st_v = torch.nn.LayerNorm(num_link)(link_flow_st)                
                demand_st_v = torch.tanh(torch.mm(demand_st_v, self.weights1_v[s]) + self.bias1_v[s])
                demand_st_v = torch.tanh(torch.mm(demand_st_v, self.weights2_v[s]) + self.bias2_v[s])
                demand_st_v = torch.tanh(torch.mm(demand_st_v, self.weights3_v[s]) + self.bias3_v[s])
                
                demand_st_t = torch.nn.LayerNorm(num_od)(od_travel_time)                
                demand_st_t = torch.tanh(torch.mm(demand_st_t, self.weights1_t[s]) + self.bias1_t[s])
                demand_st_t = torch.tanh(torch.mm(demand_st_t, self.weights2_t[s]) + self.bias2_t[s])
                demand_st_t = torch.tanh(torch.mm(demand_st_t, self.weights3_t[s]) + self.bias3_t[s])
                
                demand_st0 = torch.cat([demand_st_v, demand_st_t], dim=1)
                demand_st0 = torch.tanh(torch.mm(demand_st0, self.weights_fusion[s]) + self.bias_fusion[s])
                
                ##############
                
                demand_st_v2 = torch.mm(demand_st, self.weights1_v2[s]) + self.bias1_v2[s]
                demand_st_v2 = F.leaky_relu(demand_st_v2, negative_slope=0.2)
#                 demand_st_v2 = torch.mm(demand_st_v2, self.weights2_v2[s]) + self.bias2_v2[s]
#                 demand_st_v2 = F.leaky_relu(demand_st_v2, negative_slope=0.2)
#                 demand_st_v2 = torch.mm(demand_st_v2, self.weights3_v2[s]) + self.bias3_v2[s]
#                 demand_st_v2 = F.leaky_relu(demand_st_v2, negative_slope=0.2)

                
                demand_st = (1 + 0.5*demand_st0) * demand_st + demand_st_v2
                
                # The second source of demand_st, obtained by evenly distributing the total demand
                
                # Assignment
                theta_st = theta[s, t+1]
                link_flow_st,link_time_st = assignment(demand_st, theta_st, path_time_st, LP_s)
                
                # Update the path time and od travel time
                path_time_st = torch.mm(link_time_st,LP_s)
                od_travel_time = torch.mean(path_time_st.reshape(batch_size, num_od, k),dim=-1)
                
                link_flow_s[t+1] = link_flow_st
                demand_s[t+1] = demand_st
                
                demand_sum_s = demand_sum_s + demand_st
            
            link_flow[s] = link_flow_s
            demand_pred[s] = demand_s
            
            demand_sum[s] = demand_sum_s
        
        att, link_flow = self.attention(demand,link_flow)
        
        # att shape: (T, batch size, 1, S)
        # demand_pred shape: (S, T, batch_size, num_od)
        demand_pred = torch.matmul(att,demand_pred.permute(1,2,0,3)).reshape(T, batch_size,-1)
        
        att = att.reshape(T, batch_size, S)
        
        return att, demand_sum, demand_pred, link_flow

## 8. Define Observed and Unobserved Links

Designate 10 links as **unobserved** (no sensor deployed). The model is trained using the loss computed only on the remaining 66 observed links, but must predict flows on all 76 links.

This setup tests the model's ability to estimate flows on sensorless links â€” a key practical requirement for traffic demand flow estimation.

In [361]:
# Randomly select unobservable link numbers
#unobserved_link = np.random.choice(num_link, 10, replace=False)+1
#observed_link = np.delete(np.arange(1,num_link+1),[unobserved_link-1])
unobserved_link = np.array([47,41,67,21,25,7,61,13,37,4])
observed_link = np.array([1,2,3,5,6,8,9,10,11,12,14,15,16,17,18,19,20,22,23,24,26,27,28,
                          29,30,31,32,33,34,35,36,38,39,40,42,43,44,45,46,48,49,50,51,52,
                          53,54,55,56,57,58,59,60,62,63,64,65,66,68,69,70,71,72,73,74,75,76])
print(unobserved_link)
print(observed_link)

[47 41 67 21 25  7 61 13 37  4]
[ 1  2  3  5  6  8  9 10 11 12 14 15 16 17 18 19 20 22 23 24 26 27 28 29
 30 31 32 33 34 35 36 38 39 40 42 43 44 45 46 48 49 50 51 52 53 54 55 56
 57 58 59 60 62 63 64 65 66 68 69 70 71 72 73 74 75 76]


## 8.1 Instantiate the MTCG Model

Create the MTCG model instance with:
- `num_link = 76`, `num_od = 96`
- Hidden layers: `[512, 512]` for ODME DNNs
- Attention hidden size: 128
- Demand DNN hidden layers: `[512, 512]`

In [362]:
# Initialize MRLN
mrln = MRLN(num_link=num_link, num_od=num_od,hidden_layer=[512,512],att_hidden_size=128, hidden_layer2=[512,512])
#             attention = True, share_theta = True)
# View training parameters
for name, parameter in mrln.named_parameters():
    print(name, parameter.shape)

theta torch.Size([4, 8, 96])
weights_d1 torch.Size([4, 96, 512])
weights_d2 torch.Size([4, 512, 512])
weights_d3 torch.Size([4, 512, 768])
bias_d1 torch.Size([4, 512])
bias_d2 torch.Size([4, 512])
bias_d3 torch.Size([4, 768])
weights1_v torch.Size([4, 76, 512])
weights2_v torch.Size([4, 512, 512])
weights3_v torch.Size([4, 512, 96])
bias1_v torch.Size([4, 512])
bias2_v torch.Size([4, 512])
bias3_v torch.Size([4, 96])
weights1_t torch.Size([4, 96, 512])
weights2_t torch.Size([4, 512, 512])
weights3_t torch.Size([4, 512, 96])
bias1_t torch.Size([4, 512])
bias2_t torch.Size([4, 512])
bias3_t torch.Size([4, 96])
weights_fusion torch.Size([4, 192, 96])
bias_fusion torch.Size([4, 96])
weights1_v2 torch.Size([4, 96, 96])
bias1_v2 torch.Size([4, 96])
attention.weights_demand torch.Size([96, 128])
attention.weights_linkflow torch.Size([8, 76, 128])


## 9. Optimizer and Hyperparameters

Configure the training setup:
- **Optimizer**: Adam with learning rate $\eta = 0.002$ and momentum parameters $\beta_1 = 0.5$, $\beta_2 = 0.999$
- **Loss functions**: MSE loss and L1 loss
- **Batch size**: 64 (randomly sampled from 400 training samples)
- **Epochs**: 1000
- **Gradient clipping**: max norm = 1 (L2 norm) for training stability

In [363]:
mse_loss = nn.MSELoss()
l1_loss = nn.L1Loss()
optimizer = torch.optim.Adam(mrln.parameters(), lr=0.002, betas=(0.5, 0.999))
batch_size = 64
n_epochs = 1000
sample_size = 400

# Dynamic output folder based on S and k
output_dir = f"Estimations (S={S} k={k})"
import os
os.makedirs(output_dir, exist_ok=True)
print(f"Results will be saved to: {output_dir}")


Results will be saved to: Estimations (S=4 k=5)


## 9.1 Loss Storage

Initialize lists to record training and test loss at each epoch for convergence monitoring.

In [364]:
# List to store training loss for each epoch
loss_train_linkflow_list = []
loss_train_demand_list = []
loss_train_vdf_list = []
loss_train_total_list = []

# List to store test loss for each epoch
loss_test_linkflow_list = []
loss_test_demand_list = []
loss_test_vdf_list = []
loss_test_total_list = []

## 10. Training Loop

Define the training function implementing the **three-component weighted loss**:

$$\text{Loss} = \mu_1 \, L_{\mathbf{V}} + \mu_2 \, L_{\mathbf{Q}} + \mu_3 \, L_D$$

where:

- $L_{\mathbf{V}}$: **Link flow loss** – computed only on observed (sensor-equipped) links $\tilde{\mathcal{A}} \subseteq \mathcal{A}$:
$$L_{\mathbf{V}} = \sum_{m=1}^{M_{\mathrm{Link}}} \sum_{t=1}^{T} \frac{1}{2}\left(\mathbf{v}_{t,\tilde{\mathcal{A}}}-\bar{\mathbf{v}}^{\,m}_{t,\tilde{\mathcal{A}}}\right) \mathbf{W}^{\mathrm{Link}}_{m} \left(\mathbf{v}_{t,\tilde{\mathcal{A}}}-\bar{\mathbf{v}}^{\,m}_{t,\tilde{\mathcal{A}}}\right)^{\!\top}$$

- $L_{\mathbf{Q}}$: **OD demand conservation** – ensures predicted total demand matches the aggregate reference:
$$L_{\mathbf{Q}} = \sum_{m=1}^{M_{\mathrm{OD}}} \frac{1}{2}\left(\sum_{t=1}^{T} \mathbf{q}_t - \bar{\mathbf{q}}^{\,m}\right) \mathbf{W}^{\mathrm{OD}}_{m} \left(\sum_{t=1}^{T} \mathbf{q}_t - \bar{\mathbf{q}}^{\,m}\right)^{\!\top}$$

- $L_D$: **VDF (Beckmann) regulariser** – encourages UE-consistent flows on all links, especially unobserved ones:
$$L_D = \frac{1}{T \, |\mathcal{A}|} \sum_{t=1}^{T} \sum_{a \in \mathcal{A}} \int_0^{v_{a,t}} t_a(x) \, dx$$

Loss weights: $\mu_1 = 1$, $\mu_2 = 0.8$, $\mu_3 = 0.01$.

The model is trained via **backpropagation through the entire computational graph**, with gradients flowing from the loss through the attention mechanism, recurrent connections, and CG layers back to all learnable parameters ($\boldsymbol{\theta}$, DNN weights, attention projections).

In [365]:
def train(n_epochs, batch_size, sample_size):
    t0 = time.time()
    for epoch in range(n_epochs):
    
        # Randomly sample batch_size number of samples
        idx = np.random.choice(sample_size, batch_size, replace=False)
        
        mrln.zero_grad()

        # Input
        demand_input = train_input[idx]
        link_flow_output = train_output[:,idx,:]
        
        # Output
        att_pred, demand_sum, demand_pred, link_flow_pred = mrln(demand_input)
        
        # link flow loss
        flow_with_sensors = link_flow_output[:,:,observed_link-1]
        flow_pred_with_sensors = link_flow_pred[:,:,observed_link-1]
        
        link_flow_loss = mse_loss(flow_pred_with_sensors,flow_with_sensors)
        
        # demand conservation loss
        demand_loss = mse_loss(demand_sum, demand_input.reshape(1,batch_size,-1).repeat(S,1,1))
        
        # vdf_loss
        vdf_estimate = link_flow_pred * free_flow_tt + 0.03*(link_flow_pred**5)*free_flow_tt/(link_capacity**4)
        vdf_loss = torch.mean(torch.sum(vdf_estimate, dim=-1))
        
        # Per-timestep OD demand loss
        demand_ref_batch = demand_reference[:T, idx, :]   # (T, batch_size, num_od)
        demand_timestep_loss = mse_loss(demand_pred, demand_ref_batch)
        
        total_loss = w1 * link_flow_loss + w2 * demand_loss + w3 * vdf_loss + w4 * demand_timestep_loss

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters = mrln.parameters(), max_norm=1, norm_type=2)
        optimizer.step()

        # ---------- Record training loss ----------
        loss_train_linkflow_list.append(link_flow_loss.item())
        loss_train_demand_list.append(demand_loss.item() + demand_timestep_loss.item())
        loss_train_vdf_list.append(vdf_loss.item())
        loss_train_total_list.append(total_loss.item())
        
        # ---------- Test phase (full test evaluation each epoch) ----------
        with torch.no_grad():
            att_pred_test, demand_sum_test, demand_test_pred, link_flow_test_pred = mrln(test_input)

            # link flow loss on test
            flow_with_sensors_test = test_output[:, :, observed_link - 1]
            flow_pred_with_sensors_test = link_flow_test_pred[:, :, observed_link - 1]
            link_flow_loss_test = mse_loss(flow_pred_with_sensors_test, flow_with_sensors_test)

            # demand conservation loss on test
            demand_loss_test = mse_loss(demand_sum_test, test_input.reshape(1, test_input.shape[0], -1).repeat(S, 1, 1))

            # vdf loss on test
            vdf_estimate_test = link_flow_test_pred * free_flow_tt + 0.03 * (link_flow_test_pred ** 5) * free_flow_tt / (link_capacity ** 4)
            vdf_loss_test = torch.mean(torch.sum(vdf_estimate_test, dim=-1))

            # Per-timestep OD demand loss on test
            demand_ref_test = demand_reference[:T, 400:, :]   # (T, 100, num_od)
            demand_timestep_loss_test = mse_loss(demand_test_pred, demand_ref_test)

            total_loss_test = w1 * link_flow_loss_test + w2 * demand_loss_test + w3 * vdf_loss_test + w4 * demand_timestep_loss_test

            # Record test loss
            loss_test_linkflow_list.append(link_flow_loss_test.item())
            loss_test_demand_list.append(demand_loss_test.item() + demand_timestep_loss_test.item())
            loss_test_vdf_list.append(vdf_loss_test.item())
            loss_test_total_list.append(total_loss_test.item())

        
        if epoch % 100 == 0 or epoch == n_epochs - 1:
            print(
                "[Epoch %d/%d][link flow: %.4f][demand: %.4f][demand_ts: %.4f][vdf: %.4f][total: %.4f][time: %.1f]"
                % (epoch, n_epochs, link_flow_loss.item(), demand_loss.item(), 
                   demand_timestep_loss.item(), vdf_loss.item(), total_loss.item(), time.time()-t0)
            )


## 10.1 Execute Training

Run the training loop for 1000 epochs. At each epoch, a random mini-batch of 64 samples is drawn from the training set, and a full evaluation is performed on the test set.

In [366]:
train(n_epochs, batch_size, sample_size)

[Epoch 0/1000][link flow: 21490232.0000][demand: 18681482.0000][demand_ts: 189140.9219][vdf: 81744144.0000][total: 37252856.0000][time: 0.3]
[Epoch 100/1000][link flow: 759929.1875][demand: 881204.4375][demand_ts: 24999.6582][vdf: 3908455.0000][total: 1503977.2500][time: 23.7]
[Epoch 200/1000][link flow: 805521.8125][demand: 4178672.0000][demand_ts: 55761.1719][vdf: 2374178.0000][total: 4172201.2500][time: 44.9]
[Epoch 300/1000][link flow: 713832.5625][demand: 735252.1875][demand_ts: 32082.4785][vdf: 3794263.2500][total: 1339976.8750][time: 68.4]
[Epoch 400/1000][link flow: 951147.1875][demand: 4547921.5000][demand_ts: 31258.4277][vdf: 4417768.0000][total: 4633662.0000][time: 91.1]
[Epoch 500/1000][link flow: 1259910.7500][demand: 2357488.2500][demand_ts: 49052.9844][vdf: 4970279.5000][total: 3195604.2500][time: 114.9]
[Epoch 600/1000][link flow: 1018160.2500][demand: 2071724.0000][demand_ts: 99575.0312][vdf: 4633722.0000][total: 2721876.7500][time: 136.7]
[Epoch 700/1000][link flow: 4

## 11. Evaluation

Evaluate the trained model on the test set (100 samples) and compute estimation errors.

In [367]:
att_pred, demand_test_sum_pred, demand_test_pred, link_flow_test_pred = mrln(test_input)

## 11.1 Compute Performance Metrics

Compute the three standard evaluation metrics for **link flows** using $M_{\mathrm{Link}}$ test samples:

$$\text{RMSE}_{\mathbf{V}} = \sqrt{\frac{1}{M_{\mathrm{Link}}\, T \, |\mathcal{A}|} \sum_{m=1}^{M_{\mathrm{Link}}} \sum_{t=1}^{T} \sum_{a\in\mathcal{A}} (v_{a,t} - \bar{v}_{a,t}^m)^2}$$

$$\text{MAE}_{\mathbf{V}} = \frac{1}{M_{\mathrm{Link}}\, T \, |\mathcal{A}|} \sum_{m=1}^{M_{\mathrm{Link}}} \sum_{t=1}^{T} \sum_{a\in\mathcal{A}} |v_{a,t} - \bar{v}_{a,t}^m|$$

$$\text{MAPE}_{\mathbf{V}} = \frac{100}{M_{\mathrm{Link}}\, T \, |\mathcal{A}|} \sum_{m=1}^{M_{\mathrm{Link}}} \sum_{t=1}^{T} \sum_{a\in\mathcal{A}} \left|\frac{v_{a,t} - \bar{v}_{a,t}^m}{\bar{v}_{a,t}^m}\right| \%$$

When time-dependent OD reference values are available, analogous metrics can be defined for OD demand estimation by replacing link flows with OD demands over $|\mathcal{W}|$ pairs and $M_{\mathrm{OD}}$ samples.

Metrics are reported separately for:
- Links with sensors (66 observed links)
- Links without sensors (10 unobserved links)
- All links (total)
- OD demand

In [368]:
Error = torch.zeros(4,3)
Error[0,0] = mse_loss(test_output[:,:,observed_link-1], link_flow_test_pred[:,:,observed_link-1])**0.5
Error[0,1] = l1_loss(test_output[:,:,observed_link-1], link_flow_test_pred[:,:,observed_link-1])
Error[0,2] = torch.mean(torch.abs(test_output[:,:,observed_link-1]-link_flow_test_pred[:,:,observed_link-1])
                        /test_output[:,:,observed_link-1])
Error[1,0] = mse_loss(test_output[:,:,unobserved_link-1], link_flow_test_pred[:,:,unobserved_link-1])**0.5
Error[1,1] = l1_loss(test_output[:,:,unobserved_link-1], link_flow_test_pred[:,:,unobserved_link-1])
Error[1,2] = torch.mean(torch.abs(test_output[:,:,unobserved_link-1]-link_flow_test_pred[:,:,unobserved_link-1])
                        /test_output[:,:,unobserved_link-1])
Error[2,0] = mse_loss(test_output, link_flow_test_pred)**0.5
Error[2,1] = l1_loss(test_output, link_flow_test_pred)
Error[2,2] = torch.mean(torch.abs(test_output-link_flow_test_pred)/test_output)

Error[3,0] = mse_loss(demand_reference[:,400:,:], demand_test_pred)**0.5
Error[3,1] = l1_loss(demand_reference[:,400:,:], demand_test_pred)
Error[3,2] = torch.mean(torch.abs(demand_reference[:,400:,:]-demand_test_pred)/demand_reference[:,400:,:])

Error = Error.detach().numpy()
Error[:,:2] = np.floor(100*Error[:,:2])/100
Error[:,2] = np.floor(10000*Error[:,2])/10000
Error = DataFrame(Error,index = ['With sensors', 'Without sensors', 'Link flow total error','OD demand'], columns = ['RMSE', 'MAE', 'MAPE'])
Error

,RMSE,MAE,MAPE
With sensors,621.260010,489.209991,0.0986
Without sensors,1009.179993,871.140015,0.1777
Link flow total error,684.969971,539.460022,0.1090
OD demand,342.529999,249.419998,0.2572


## 12. Export Results

Export estimation results and loss trajectories to Excel files for post-processing and figure generation.

In [369]:
import pandas as pd

# 1) Export link_flow_estimation.xlsx
# -----------------------------------
# Reshape (T, 100, 76) to align with ground truth
est_flow = link_flow_test_pred.detach().cpu().numpy().reshape(-1)
obs_flow = test_output.detach().cpu().numpy().reshape(-1)

df_flow = pd.DataFrame({
    "Estimation": est_flow,
    "Observation": obs_flow
})
df_flow.to_excel(f"{output_dir}/link_flow_estimation.xlsx", index=False)
print(f"{output_dir}/link_flow_estimation.xlsx has been saved!")

# 2) Export od_demand_estimation.xlsx
# -----------------------------------
est_demand = demand_test_pred.detach().cpu().numpy().reshape(-1)
ref_demand = demand_reference[:,400:,:].detach().cpu().numpy().reshape(-1)

df_demand = pd.DataFrame({
    "Estimation": est_demand,
    "Reference": ref_demand
})
df_demand.to_excel(f"{output_dir}/od_demand_estimation.xlsx", index=False)
print(f"{output_dir}/od_demand_estimation.xlsx has been saved!")

Estimations (S=4 k=5)/link_flow_estimation.xlsx has been saved!
Estimations (S=4 k=5)/od_demand_estimation.xlsx has been saved!


## 12.2 Export Per-Template Results and Attention Weights

Save attention weights and per-template link flow estimations for analysis in `Plot.ipynb`.
When `S=1`, these reduce to the fused output; for `S>1`, they enable per-template comparison.

In [370]:
import numpy as np

# --- Extract per-template link flows (before attention fusion) ---
# Re-run forward pass internals to capture pre-fusion per-template flows
with torch.no_grad():
    # The model's forward() fuses link_flow via attention.
    # We hook into the attention layer to capture the pre-fusion tensor.
    _pre_fusion_flow = {}

    def _hook_fn(module, input_args, output):
        # Attention.forward receives (demand, X) where X shape: (S, T, batch, num_link)
        _pre_fusion_flow["link_flow_per_template"] = input_args[1].detach().cpu().numpy()

    hook = mrln.attention.register_forward_hook(_hook_fn)
    att_out, _, _, _ = mrln(test_input)
    hook.remove()

    # Per-template link flows: shape (S, T, 100, 76)
    link_flow_per_template = _pre_fusion_flow["link_flow_per_template"]
    print(f"Per-template link flow shape: {link_flow_per_template.shape}")

    # Attention weights: shape (T, 100, 1, S) -> save as (T, 100, S)
    att_weights = att_out.detach().cpu().numpy().reshape(T, -1, S)
    print(f"Attention weights shape: {att_weights.shape}")

# Save per-template link flows
np.savez(f"{output_dir}/per_template_results.npz",
         link_flow_per_template=link_flow_per_template,
         attention_weights=att_weights)
print(f"{output_dir}/per_template_results.npz has been saved!")

# Also save as Excel for easier inspection (one sheet per template)
with pd.ExcelWriter(f"{output_dir}/link_flow_per_template.xlsx") as writer:
    for s in range(S):
        df_s = pd.DataFrame(link_flow_per_template[s].reshape(-1, num_link),
                            columns=[f"link_{a+1}" for a in range(num_link)])
        df_s.to_excel(writer, sheet_name=f"template_{s+1}", index=False)
print(f"{output_dir}/link_flow_per_template.xlsx has been saved!")

Per-template link flow shape: (4, 8, 100, 76)
Attention weights shape: (8, 100, 4)
Estimations (S=4 k=5)/per_template_results.npz has been saved!
Estimations (S=4 k=5)/link_flow_per_template.xlsx has been saved!


## 12.1 Export Loss Trajectories

Save training and test loss values across epochs to Excel for plotting convergence curves.

In [371]:
# Generate DataFrame for training loss
df_train = pd.DataFrame({
    "link_flow": loss_train_linkflow_list,
    "demand": loss_train_demand_list,
    "vdf": loss_train_vdf_list,
    "total": loss_train_total_list
})
df_train.to_excel(f"{output_dir}/loss_train.xlsx", index=False)
print(f"{output_dir}/loss_train.xlsx has been saved.")

# Generate DataFrame for test loss
df_test = pd.DataFrame({
    "link_flow": loss_test_linkflow_list,
    "demand": loss_test_demand_list,
    "vdf": loss_test_vdf_list,
    "total": loss_test_total_list
})
df_test.to_excel(f"{output_dir}/loss_test.xlsx", index=False)
print(f"{output_dir}/loss_test.xlsx has been saved.")

Estimations (S=4 k=5)/loss_train.xlsx has been saved.
Estimations (S=4 k=5)/loss_test.xlsx has been saved.
